# F: Full pipeline run (detector → explain → guidebook → rewrite → reviewers → second detector)

This notebook runs **every part of the project together**, on Amen's 50 test sentences, and measures the result
with a detector that had no part in the rewrite.

```
chatbot answer
   │
   ▼
1 Detector (XLM-R) ──► unbiased: left as it is
   │                └─► uncertain: to a human
   ▼ biased
2 Explainer (Captum): which words made it biased?
   │
   ▼
3 Guidebook search (RAG): the 3 rules that fit those words
   │
   ▼
4 Rewriter (Llama 3.1 8B) ◄───────────── feedback (up to 3 rounds)
   │                                         ▲
   ▼                                         │
5 Fact check: no new numbers or names        │
6 Reviewer A (Palestinian side)              │
  Reviewer B (Israeli side)                  │
   │                                         │
   ▼                                         │
7 Evaluator: approve │ retry ────────────────┘ │ escalate (reviewers disagree) │ block (after round 3)

8 Second detector (mDeBERTa, other split): scores the original and the final answer   ← evaluation only
```

| Step | Code | Owner |
|---|---|---|
| 1–2 Detector, explainer | `src/detector.py` | Iroda |
| 3 Guidebook search | `src/rag.py`, `data/guidebook.md` | Yasa, Amen |
| 4 Rewriter | `src/rewriter.py` (Llama 3.1 8B, 4-bit) | Madina |
| 5–7 Fact check, reviewers, evaluator | `src/pipeline.py`, `src/reviewers.py` | Madina, Iroda |
| 8 Second detector | `src/second_detector.py` | Iroda |

**Before you start**
- Runtime → Change runtime type → **T4 GPU**.
- Colab Secrets (key icon on the left): `HF_TOKEN` with access to `meta-llama/Llama-3.1-8B-Instruct`, "Notebook access" on.
  Never type the token into a cell: the repo is public.
- Both trained detectors on Drive (from `A_detector`): `debias/detector/xlm-roberta-base_seed44/model` and
  `debias/detector/mdeberta-v3-base_seed42/model`.
- Time: about 30–60 s per sentence, so roughly 30–45 minutes for all 50. Progress is saved after every sentence;
  if Colab disconnects, run the setup cell and the run cell again and it continues where it stopped.

In [1]:
import os
os.environ["DEBIAS_ARTIFACTS"] = "/content/drive/.shortcut-targets-by-id/1tkCMcppW2cr3O3Voim0u31joay_zy1WB/debias"

In [4]:
# TEAM SETUP CELL
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/iibrohim-hash/debias.git"
BRANCH = "main"  # use your feature branch while developing, e.g. "iroda/detector"

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/debias")
elif IN_KAGGLE:
    REPO_DIR = Path("/kaggle/working/debias")
else:  # local Jupyter: notebook is inside repo/notebooks
    REPO_DIR = Path.cwd().parent if (Path.cwd().parent / "src" / "config.py").exists() else Path.cwd()

if IN_COLAB or IN_KAGGLE:
    if not (REPO_DIR / "src").exists():
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "-q"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

try:  # autoreload is broken on some Colab images (Python 3.13: no 'imp' module)
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    print("autoreload not available: restart the runtime after editing src/")

from src import config, utils, contract
utils.set_seed(config.SEED)
if utils.get_hf_token():  # from Colab/Kaggle secrets; never type a token here (public repo)
    os.environ["HF_TOKEN"] = utils.get_hf_token()
ARTIFACTS = utils.artifacts_dir()
print("repo:", REPO_DIR, "| artifacts:", ARTIFACTS)
utils.print_env()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
autoreload not available: restart the runtime after editing src/
repo: /content/debias | artifacts: /content/drive/.shortcut-targets-by-id/1tkCMcppW2cr3O3Voim0u31joay_zy1WB/debias
runtime=colab  python=3.13.16  gpu=Tesla T4  commit=ba08bfd
  torch                  2.14.1+cu130
  transformers           5.23.0
  datasets               5.2.0
  accelerate             1.15.0
  bitsandbytes           0.50.2
  sentence-transformers  6.3.0
  faiss-cpu              1.16.0
  captum                 0.9.0
  scikit-learn           1.6.1
  pandas                 2.2.3
  numpy                  2.4.6


## 1. Check that every part is in place

Nothing is loaded yet. This cell only checks that the files, the guidebook, the test sentences and the Llama
licence are there, so the run does not fail 20 minutes in.

In [5]:
from src import detector, second_detector, rag, rewriter, reviewers, pipeline
import pandas as pd

MAIN_DIR = ARTIFACTS / "detector" / "xlm-roberta-base_seed44" / "model"    # best seed by validation
SECOND_DIR = ARTIFACTS / "detector" / second_detector.DEFAULT_RUN / "model"  # mDeBERTa, other split

for name, path in [("main detector (XLM-R)", MAIN_DIR), ("second detector (mDeBERTa)", SECOND_DIR)]:
    assert path.exists(), f"{name} not found at {path}: run A_detector first"
    print(f"OK  {name:28s} {path}")
detector.set_model_path(MAIN_DIR)
second_detector.set_model_path(SECOND_DIR)

entries = rag.load_entries(config.GUIDEBOOK_PATH)
print(f"OK  guidebook                    {len(entries)} rules")

tests = pd.read_csv(config.TEST_SENTENCES_PATH)
tests["split"] = ["dev" if i <= 19 else "held-out" for i in tests["id"]]
print(f"OK  test sentences               {len(tests)} ({(tests.split == 'dev').sum()} dev, {(tests.split == 'held-out').sum()} held-out)")

print("    Llama access:", rewriter.check_model_access())
print("    reviewer prompt version:", reviewers.prompt_version())
print("    rewriter prompt versions:", rewriter.prompt_versions())

OK  main detector (XLM-R)        /content/drive/.shortcut-targets-by-id/1tkCMcppW2cr3O3Voim0u31joay_zy1WB/debias/detector/xlm-roberta-base_seed44/model
OK  second detector (mDeBERTa)   /content/drive/.shortcut-targets-by-id/1tkCMcppW2cr3O3Voim0u31joay_zy1WB/debias/detector/mdeberta-v3-base_seed42/model
OK  guidebook                    25 rules
OK  test sentences               50 (19 dev, 31 held-out)


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

    Llama access: iroda-ibr
    reviewer prompt version: 00d85d1a00
    rewriter prompt versions: {'rewriter_system.txt': '18e8a480bf', 'rewriter_user.txt': '504a302b2f', 'rewriter_retry.txt': '0856859d67'}


**Dev vs held-out.** Sentences 1–19 were used while writing the guidebook and the rewriter prompt (dev).
Sentences 20–50 were added later without changing any rule (held-out). Numbers on the held-out sentences are the
honest ones; the two groups are reported separately below.

## 2. Load all models once

Four models share one T4 GPU (16 GB). Llama is loaded **once** and used by both the rewriter and the two reviewers.

In [8]:
import time, torch

def timed(label, fn):
    start = time.perf_counter(); out = fn()
    print(f"{label:34s} {time.perf_counter() - start:6.1f} s"); return out

timed("main detector (XLM-R)", lambda: detector.detect("warm up"))
timed("guidebook search (MiniLM + FAISS)", lambda: rag.retrieve("warm up"))
_, _, model_id = timed("Llama 3.1 8B, 4-bit", rewriter.get_model)
timed("second detector (mDeBERTa)", lambda: second_detector.scores(["warm up"]))
assert model_id == config.REWRITER_MODEL, model_id
print("rewriter and reviewers use:", model_id)
print("main detector thresholds:  ", detector.thresholds())
print("second detector thresholds:", second_detector.thresholds())
if torch.cuda.is_available():
    print(f"GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB of "
          f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

main detector (XLM-R)                 0.0 s
guidebook search (MiniLM + FAISS)     0.0 s
Llama 3.1 8B, 4-bit                   0.0 s
second detector (mDeBERTa)            0.0 s
rewriter and reviewers use: meta-llama/Llama-3.1-8B-Instruct
main detector thresholds:   {'biased': 0.58, 'uncertain_low': 0.12108474969863892, 'val_macro_f1': 0.7201166024695436}
second detector thresholds: {'biased': 0.55, 'uncertain_low': 0.12669998407363892, 'val_macro_f1': 0.714033417537521}
GPU memory in use: 8.4 GB of 15.6 GB


In [7]:
import shutil, os
run = ARTIFACTS / "detector" / "mdeberta-v3-base_seed42"
out, ckpt = run / "model", run / "checkpoints" / "checkpoint-846 (1)"
print("model folder now:", sorted(p.name for p in out.iterdir()))
print("checkpoint file:", f"{(ckpt / 'model.safetensors').stat().st_size / 1e6:.0f} MB")

if not (out / "model.safetensors").exists():
    shutil.copyfile(ckpt / "model.safetensors", out / "model.safetensors")
    os.sync()
print("model.safetensors:", f"{(out / 'model.safetensors').stat().st_size / 1e6:.0f} MB")

second_detector.set_model_path(out)
print("scores:", second_detector.scores(["warm up", "Hamas terrorists butchered families."]))

model folder now: ['config.json', 'thresholds.json', 'tokenizer.json', 'tokenizer_config.json']
checkpoint file: 1115 MB
model.safetensors: 1115 MB


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

scores: [0.15719300508499146, 0.9665966033935547]


## 3. One sentence, step by step

Before running all 50, follow one sentence through every step. `force=True` sends it through the rewrite even if
the detector does not call it biased; it is used **only here**, to see the whole chain. The full run in section 4
does not force anything.

In [9]:
def rule_ids(trace):
    return [r.split()[1] for r in trace.get("rules", [])]

def show(result, trace, conflict=config.DEFAULT_CONFLICT):
    d = trace["detect"]
    print("INPUT:", trace["input"])
    print(f"1 DETECTOR    {d['status']}  score={d['score']:.2f}  label={d['label']}"
          + ("   (forced through)" if trace["forced"] and d["status"] != "biased" else ""))
    if "spans" not in trace:
        print("  stops here:", result["decision"], result.get("reason", ""))
    else:
        print(f"2 EXPLAINER   flagged words: {trace['spans']}")
        print(f"3 GUIDEBOOK   rules: {rule_ids(trace)}")
        for rd in trace["rounds"]:
            rw, f = rd["rewrite"], rd["fact_check"]
            print(f"--- round {rd['round']} ---")
            if rd["feedback_in"]:
                print("  feedback from the last round:\n   ", rd["feedback_in"].replace("\n", "\n    "))
            print(f"4 REWRITER    {rw['revised']}")
            for c in rw["changes"]:
                print(f'                "{c["original"]}" -> "{c["revised"]}"  ({c["reason"]})')
            print(f"5 FACT CHECK  ok={f['ok']}  new numbers={f['new_numbers']}  new names={f['new_names']}")
            for p, r in rd["reviews"].items():
                problems = "; ".join(f'"{i["phrase"]}": {i["problem"]}' for i in r["issues"]) or "none"
                print(f"6 REVIEWER    {p:12s} fair={r['fair']}  problems: {problems}")
            print(f"7 EVALUATOR   {rd['decision']}")
    before, after = second_detector.scores([trace["input"], result["final"]], conflict)
    print(f"8 SECOND DET. before {before:.2f} ({second_detector.status(before)}) -> "
          f"after {after:.2f} ({second_detector.status(after)})")
    print("FINAL:", result["final"])
    print(result["change_note"])
    print("seconds:", trace["seconds"])

result, trace = pipeline.run_detailed(tests.loc[0, "biased"], tests.loc[0, "conflict"], force=True, log=False)
show(result, trace)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


INPUT: "Friends" actor David Schwimmer called out some advocates against sexual violence, arguing that they ignored reports of Hamas terrorists' sex crimes on Oct. 7.
1 DETECTOR    biased  score=0.99  label=against_SIDE_B
2 EXPLAINER   flagged words: ['terrorists', 'advocates', 'actor', 'crimes', 'sexual']
3 GUIDEBOOK   rules: ['R01', 'R02', 'R04', 'R20']
--- round 1 ---
4 REWRITER    The "Friends" actor David Schwimmer called out some advocates against violence, arguing that they ignored reports of Hamas members' sex offenses on Oct. 7.
                "terrorists'" -> "members'"  (R02: loaded label for group)
                "advocates against sexual violence" -> "advocates against violence"  (R20: contested label)
                "sex crimes" -> "sex offenses"  (R20: contested label)
                "sexual" -> ""  (R20: contested label)
5 FACT CHECK  ok=False  new numbers=[]  new names=['The']
6 REVIEWER    palestinian  fair=False  problems: "Hamas members'": R02: loaded label for 

## 4. Run all test sentences

Settings:
- `SPLITS`: which sentences to run. Start with `QUICK = True` (3 sentences) to check the timing, then set it to
  `False`.
- Each sentence goes through the **real** pipeline: no `force`. If the detector says "unbiased" or "uncertain",
  the sentence stops there, exactly as it would for a real chatbot answer.
- Results are appended to `full_pipeline_rows.csv` after every sentence (on Drive); a rerun skips the ids that are
  already done. Delete the file (or set `RESTART = True`) to start over, e.g. after a prompt change.
- Every sentence's full trace also goes to `audit_log.jsonl`.

In [ ]:
QUICK = True                      # True: first 3 sentences only, to check the timing
SPLITS = ("dev", "held-out")      # or ("held-out",) for the honest numbers only
RESTART = False                   # True: forget earlier results and run everything again

OUT = ARTIFACTS / "full_pipeline_rows.csv"
if RESTART and OUT.exists():
    OUT.unlink()
done = set(pd.read_csv(OUT)["id"]) if OUT.exists() else set()

todo = tests[tests["split"].isin(SPLITS) & ~tests["id"].isin(done)]
if QUICK:
    todo = todo.head(3)
print(f"{len(done)} already done, {len(todo)} to run")

for n, t in enumerate(todo.itertuples(), 1):
    result, trace = pipeline.run_detailed(t.biased, t.conflict)
    d = trace["detect"]
    s_orig, s_final, s_ref = second_detector.scores([t.biased, result["final"], t.neutral], t.conflict)
    rounds = trace["rounds"]
    row = {
        "id": t.id, "split": t.split, "conflict": t.conflict,
        "detector_status": d["status"], "detector_score": round(d["score"], 3),
        "flagged": ", ".join(trace.get("spans", [])),
        "rules": " ".join(rule_ids(trace)), "expected_rules": t.expected_rules,
        "decision": result["decision"], "rounds": len(rounds),
        "n_changes": len(result["changes"]),
        "last_rewrite": rounds[-1]["rewrite"]["revised"] if rounds else "",
        "last_fair": " ".join(f"{p}={r['fair']}" for p, r in rounds[-1]["reviews"].items()) if rounds else "",
        "original": t.biased, "final": result["final"], "reference_neutral": t.neutral,
        "second_before": round(s_orig, 3), "second_after": round(s_final, 3), "second_reference": round(s_ref, 3),
        "seconds": trace["seconds"]["total"],
        "reviewer_prompt": reviewers.prompt_version(), "commit": utils.git_commit(),
    }
    pd.DataFrame([row]).to_csv(OUT, mode="a", header=not OUT.exists(), index=False)
    print(f"[{n}/{len(todo)}] id {t.id:>2}: {d['status']:9s} -> {result['decision']:9s} "
          f"second detector {s_orig:.2f} -> {s_final:.2f}  ({row['seconds']:.0f} s)")

runs = pd.read_csv(OUT)
print(f"\n{len(runs)} of {len(tests)} sentences done. Set QUICK = False and rerun this cell for the rest.")

0 already done, 3 to run
[1/3] id  1: biased    -> escalate  second detector 0.98 -> 0.98  (41 s)


## 5. Results

**What each number means**
- **Detector verdicts**: how the main detector sorted the 50 sentences. All of them were written as biased, so
  "unbiased" here is a miss and "uncertain" means a human would be asked.
- **Decisions**: what the pipeline did with each sentence.
  `approve` = rewrite accepted; `escalate` = reviewers disagreed (or the detector was uncertain), a human decides;
  `block` = no fair rewrite after 3 rounds; `unchanged` = the detector called it unbiased.
- **Second detector**: s = 1 − P(unbiased) from mDeBERTa, before and after. A drop means the bias went down by
  the judgement of a model that never saw the rewrite step. `reference` is Amen's hand-written neutral version:
  the best we could hope for.

In [ ]:
import numpy as np

runs = pd.read_csv(OUT)
rng = np.random.default_rng(config.SEED)

def mean_ci(x, n=config.BOOTSTRAP_SAMPLES):
    x = np.asarray(x, float)
    if len(x) < 2:
        return f"{x.mean():.3f}" if len(x) else "-"
    boots = [rng.choice(x, len(x)).mean() for _ in range(n)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return f"{x.mean():.3f} [{lo:.3f}, {hi:.3f}]"

def biased_share(scores):
    return f"{np.mean([second_detector.status(s) == 'biased' for s in scores]):.2f}"

summary = {}
for split, g in [("all", runs)] + list(runs.groupby("split")):
    summary[split] = {
        "sentences": len(g),
        "detector: biased / uncertain / unbiased": " / ".join(
            str((g.detector_status == s).sum()) for s in ("biased", "uncertain", "unbiased")),
        "approve / escalate / block / unchanged": " / ".join(
            str((g.decision == s).sum()) for s in ("approve", "escalate", "block", "unchanged")),
        "mean rounds (rewritten only)": round(g.loc[g.rounds > 0, "rounds"].mean(), 2),
        "second det. score before": mean_ci(g.second_before),
        "second det. score after": mean_ci(g.second_after),
        "second det. score, reference": mean_ci(g.second_reference),
        "drop (before - after)": mean_ci(g.second_before - g.second_after),
        "share 'biased' before / after / reference": " / ".join(
            biased_share(g[c]) for c in ("second_before", "second_after", "second_reference")),
        "seconds per sentence": round(g.seconds.mean(), 1),
    }
summary_df = pd.DataFrame(summary)
summary_df

In [ ]:
# Only the approved rewrites: did the ones we would actually show users get less biased?
ok = runs[runs.decision == "approve"]
print(f"approved: {len(ok)}")
if len(ok):
    print("second detector, approved only: before", mean_ci(ok.second_before), "| after", mean_ci(ok.second_after))
    print("rule overlap with Amen's expected rules:",
          f"{np.mean([bool(set(str(r).split()) & set(str(e).split())) for r, e in zip(ok.rules, ok.expected_rules)]):.2f}")

utils.save_result("full_pipeline", {k: {kk: str(vv) for kk, vv in v.items()} for k, v in summary.items()},
                  extra={"reviewer_prompt": reviewers.prompt_version(), "rewriter_prompts": rewriter.prompt_versions(),
                         "rewriter_model": model_id, "n_rows": len(runs)})
print("saved to", ARTIFACTS / "results")

## 6. Look at the sentences

Numbers hide the interesting part. The table lists every sentence; below it, the full trace of one sentence for
each decision type is printed again so you can see *why* it ended that way.

In [ ]:
pd.set_option("display.max_colwidth", 110)
runs[["id", "split", "detector_status", "decision", "rounds", "original", "final",
      "second_before", "second_after", "last_fair"]]

In [ ]:
# One example of each decision, rerun with the full printout (same models, greedy decoding, so same result).
for decision in ("approve", "escalate", "block"):
    ex = runs[runs.decision == decision]
    if ex.empty:
        continue
    t = tests[tests.id == ex.iloc[0]["id"]].iloc[0]
    print("=" * 30, decision.upper(), f"(id {t.id})", "=" * 30)
    show(*pipeline.run_detailed(t.biased, t.conflict, log=False), conflict=t.conflict)
    print()

## How to read the results

- **Second detector drop** is the main number for the report: a clear drop on the **held-out** sentences, close to
  the reference score, means the rewrite removes bias that a separate model can see.
- **Escalate** is high when the reviewers disagree in round 1. An unfair rewrite is usually flagged only by the
  harmed side's reviewer, so with today's rule it goes to a human instead of getting another round. If escalations
  dominate, this is the team decision from the Oversight slide (retry first, escalate after round 3).
- **Unbiased / uncertain from the main detector** on sentences that are biased by design shows what the detector
  misses: those never reach the rewriter.
- **Block** means 3 rounds were not enough: read the reviewer problems in the trace; they show which guidebook rule
  the rewriter keeps breaking.
- **Caveat.** Some test sentences come from FigNews posts, which both detectors may have seen during training.
  Their scores before the rewrite can therefore be higher than for new text; the before/after *change* is still
  fair because both scores come from the same model.
- Every run is saved with the prompt versions, the model id and the commit, so numbers from different runs can be
  compared or traced back.